# Pre-Processing Data (DataFrame Approach)
In this section, we will walk through the core steps of preparing data for machine learning, strictly maintaining a Pandas DataFrame structure for better readability and column tracking. The steps include handling missing values, encoding categorical variables, feature selection, splitting the dataset, and feature scaling. Each step is crucial for ensuring that our machine learning models can learn effectively from the data and make accurate predictions on unseen data. By following these pre-processing steps, we can enhance the performance and reliability of our machine learning models.

### 1. Importing Libraries

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

### 2. Importing Dataset
Instead of converting the features to a NumPy array with `.iloc[:, :-1].values`, we will drop the target column to create `X` and isolate the target column to create `y`. This keeps our column names fully intact.

In [ ]:
# Load the dataset
dataset = pd.read_csv('../data/01_preprocessing_data.csv')

# Separate Features (X) and Target (y)
X = dataset.drop(columns=['Purchased']) 
y = dataset['Purchased']

print("Features (X):")
print(X.head())

Features (X):
   Country   Age   Salary
0   France  44.0  72000.0
1    Spain  27.0  48000.0
2  Germany  30.0  54000.0
3    Spain  38.0  61000.0
4  Germany  40.0      NaN


### 3. Missing Values
We will use `SimpleImputer` to fill missing data. By passing specifically named columns back into the DataFrame, we avoid index-slicing errors.

In [4]:
# Define which columns are numerical and might have missing values
numeric_cols = ['Age', 'Salary']

# Create the imputer object
imputer = SimpleImputer(strategy='mean')

# Fit and transform the specific columns, replacing the old columns in the DataFrame
X[numeric_cols] = imputer.fit_transform(X[numeric_cols])

print("X after handling missing values:")
print(X)

X after handling missing values:
   Country        Age        Salary
0   France  44.000000  72000.000000
1    Spain  27.000000  48000.000000
2  Germany  30.000000  54000.000000
3    Spain  38.000000  61000.000000
4  Germany  40.000000  63777.777778
5   France  35.000000  58000.000000
6    Spain  38.777778  52000.000000
7   France  48.000000  79000.000000
8  Germany  50.000000  83000.000000
9   France  37.000000  67000.000000


### 4. Data Transformation
We need to handle our categorical data. 
1. **One-Hot Encoding:** We will use Pandas' built-in `get_dummies()` for the 'Country' column. It is highly readable and automatically updates our DataFrame headers.
2. **Label Encoding:** We will use `LabelEncoder` for the target variable ('Purchased') to turn 'Yes'/'No' into 1/0.

In [5]:
# 4.1 One-Hot Encoding for Features
# By default, pd.get_dummies converts to boolean (True/False). 
# Adding astype(int) converts them to 1 and 0.
X = pd.get_dummies(X, columns=['Country']).astype(int)

# 4.2 Label Encoding for Target Variable
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("X after One-Hot Encoding:")
print(X.head())
print("\ny after Label Encoding:")
print(y)

X after One-Hot Encoding:
   Age  Salary  Country_France  Country_Germany  Country_Spain
0   44   72000               1                0              0
1   27   48000               0                0              1
2   30   54000               0                1              0
3   38   61000               0                0              1
4   40   63777               0                1              0

y after Label Encoding:
[0 1 0 0 1 1 0 1 0 1]


### 5. Splitting Dataset into Training and Testing Set
We split the data before scaling to completely prevent data leakage. The model should never "see" the test data during its preparation phases.

In [6]:
# Perform the split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print("Training Features (X_train):")
print(X_train)

Training Features (X_train):
   Age  Salary  Country_France  Country_Germany  Country_Spain
6   38   52000               0                0              1
4   40   63777               0                1              0
0   44   72000               1                0              0
3   38   61000               0                0              1
1   27   48000               0                0              1
7   48   79000               1                0              0
8   50   83000               0                1              0
5   35   58000               1                0              0


### 6. Feature Scaling
We scale only the numerical columns (`Age` and `Salary`). We explicitly do not scale the dummy variables (the new `Country` columns) because they are already perfectly bounded between 0 and 1. 

**Crucial Note:** We use `fit_transform` on the training set, but ONLY `transform` on the test set. Because the test set must be scaled using the same parameters (mean and standard deviation) as the training set to avoid data leakage and ensure that the model's performance is evaluated on truly unseen data.

In [7]:
# Initialize the scaler
sc = StandardScaler()

# Fit and transform on the training set's numerical columns
X_train[numeric_cols] = sc.fit_transform(X_train[numeric_cols])

# Transform the testing set's numerical columns using the SAME scaler
X_test[numeric_cols] = sc.transform(X_test[numeric_cols])

print("X_train after Feature Scaling:")
print(X_train)
print("\nX_test after Feature Scaling:")
print(X_test)

X_train after Feature Scaling:
        Age    Salary  Country_France  Country_Germany  Country_Spain
6 -0.289430 -1.078117               0                0              1
4  0.000000 -0.070190               0                1              0
0  0.578860  0.633570               1                0              0
3 -0.289430 -0.307858               0                0              1
1 -1.881294 -1.420454               0                0              1
7  1.157719  1.232661               1                0              0
8  1.447149  1.574998               0                1              0
5 -0.723575 -0.564611               1                0              0

X_test after Feature Scaling:
        Age    Salary  Country_France  Country_Germany  Country_Spain
2 -1.447149 -0.906948               0                1              0
9 -0.434145  0.205649               1                0              0


This is perhaps the single most important rule to understand to truly master Machine Learning data pipelines. Getting this wrong leads to a phenomenon called **Data Leakage**, which will make your model look like a genius in testing, but fail miserably in the real world. 

To understand why we do this, let's break down exactly what these two words mean under the hood, and then look at a real-world analogy.

---

### 1. The Definitions
Whenever you use a preprocessing tool in Scikit-Learn (like `StandardScaler` or `SimpleImputer`), it relies on internal math.

* **`fit()`**: This means **"Calculate the parameters."** * If you are using a `StandardScaler`, `fit()` calculates the **Mean** ($\mu$) and **Standard Deviation** ($\sigma$) of the data you give it.
    * If you are using `SimpleImputer` (with strategy='mean'), `fit()` calculates the average value of the column.
* **`transform()`**: This means **"Apply the math."** * It takes the parameters learned during `fit()` and actually changes the data. For scaling, it applies the formula: $z = \frac{x - \mu}{\sigma}$
* **`fit_transform()`**: This does both steps simultaneously. It calculates the parameters and immediately applies them to that same batch of data.

---

### 2. The Exam Analogy
Think of building an ML model like preparing a student for a final exam.

* **Training Data** = The practice tests the student uses to study.
* **Test Data** = The actual Final Exam.

If you use `fit_transform()` on your entire dataset *before* you split it, or if you `fit()` the scaler on your test data, you are calculating the Mean and Standard Deviation using the Final Exam's data. 

You have just let the student peek at the Final Exam while they were studying. They will score a 100% on the test, but they haven't actually learned the material; they just memorized the test's specific quirks. This is **Data Leakage**.

---

### 3. The Mathematical Reality
Let's say you are predicting house prices, and you are scaling the "Square Footage" feature. 

**What happens on the Training Set:**
1.  You run `X_train = sc.fit_transform(X_train)`.
2.  The scaler looks at your training houses and calculates: "The average house here is 2,000 sq ft." ($\mu = 2000$).
3.  It scales the training data based on that 2,000 sq ft baseline.

**What happens on the Test Set:**
1.  You run `X_test = sc.transform(X_test)`. 
2.  Notice there is no `fit` here! The scaler **remembers** the baseline from the training data ($\mu = 2000$).
3.  When it sees a 2,500 sq ft house in the test set, it evaluates it strictly against the 2,000 sq ft average it learned during training.

**What if you made the mistake of using `fit_transform` on the Test Set?**
The scaler would look at the test data and calculate a brand *new* average (let's say $\mu = 2,200$). Now, your training data was scaled using a baseline of 2,000, but your test data is scaled using a baseline of 2,200. The scale is completely broken, and the model's predictions will be mathematical nonsense.

### Summary Table

| Dataset | Method to Use | What the Scaler Does |
| :--- | :--- | :--- |
| **Training Set** | `fit_transform()` | Calculates the math parameters ($\mu, \sigma$) **AND** scales the data. |
| **Testing Set** | `transform()` | **ONLY** scales the data, strictly using the parameters it memorized from the Training Set. |
| **New Real-World Data**| `transform()` | **ONLY** scales the data, using the original Training Set parameters. |

---

When you deploy a model into a live production environment, it will only receive data one row at a time. You *cannot* calculate a meaningful average on a single new row of data. The model *must* rely on the context it learned during training. 

Does the distinction between calculating the rules (`fit`) and applying the rules (`transform`) make sense now? If so, we are ready to move from data prep to actually training a model to make predictions!

### 7. Conclusion
By utilizing DataFrames throughout the entire preprocessing pipeline, we maintain complete visibility over our feature names, prevent index-shifting bugs, and write cleaner, more Pythonic code. Our data is now fully cleaned, encoded, scaled, and ready to be fed into a Machine Learning model.